### Save all PBP point  data

In [15]:
import pandas as pd
import numpy as np
import json
import os

pd.set_option('display.max_columns', None)
pd.set_option("display.max_rows", None)
import sys
sys.path.append('../src')

from process import get_match_info, get_match_point_level_info
from search_utils import get_unique_player_ids, normalize_name, get_all_players_info, get_all_players_info_df


In [16]:
def create_directory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

tournament_list = ['australian_open', 'roland_garros']
league_list = ['atp', 'wta']

players_matches_catalogue_folder = 'players_matches_catalogue'
create_directory(players_matches_catalogue_folder)

catalogue_csv_path = os.path.join(players_matches_catalogue_folder, 'catalogue_all_matches_available.csv')

if os.path.exists(catalogue_csv_path):
    os.remove(catalogue_csv_path)

for tournament in tournament_list:
    for league in league_list:
        dir_name = '../all_json/' + tournament + '/' + league + '/'
        csv_save_path = os.path.join('pbp_data_csvs', tournament)
        create_directory(csv_save_path)
        csv_save_path = os.path.join(csv_save_path, league)
        create_directory(csv_save_path)
        
        for filename in os.listdir(dir_name):
            if filename.endswith(".json"):
                print(filename)
                with open(os.path.join(dir_name, filename)) as file_name:
                    tracking_data_json = json.load(file_name)
                    file_year = [int(s) for s in filename.split('_') if s.isdigit()]
                    
                    match_id = filename[17:-19]
                    match_pbp_data =  get_match_point_level_info(tracking_data_json)
                    hasYearInMatchId = ""
                    if tournament == "roland_garros":
                        hasYearInMatchId = str(file_year[0]) + '_'
                        
                    save_data_filename = league + '_' + tournament +  '_' + hasYearInMatchId + match_id + '_pbp' + '.csv'
                    save_data_path = os.path.join(csv_save_path, save_data_filename)
                    
                    #Add scores
                    
                    match_pbp_data.to_csv(save_data_path, index=False)
                    
                    match_catalogue_data = get_match_info(tracking_data_json, tournament, league, match_id, str(file_year[0]))
                    
                    match_catalogue_data.to_csv(catalogue_csv_path, mode='a', index=False, header=not os.path.exists(catalogue_csv_path))

            else:
                continue


atp_AO_Open_year_2020_MS101_tracking_data.json
atp_AO_Open_year_2020_MS102_tracking_data.json
atp_AO_Open_year_2020_MS103_tracking_data.json
atp_AO_Open_year_2020_MS104_tracking_data.json
atp_AO_Open_year_2020_MS105_tracking_data.json
atp_AO_Open_year_2020_MS106_tracking_data.json
atp_AO_Open_year_2020_MS107_tracking_data.json
atp_AO_Open_year_2020_MS108_tracking_data.json
atp_AO_Open_year_2020_MS109_tracking_data.json
atp_AO_Open_year_2020_MS110_tracking_data.json
atp_AO_Open_year_2020_MS111_tracking_data.json
atp_AO_Open_year_2020_MS112_tracking_data.json
atp_AO_Open_year_2020_MS113_tracking_data.json
atp_AO_Open_year_2020_MS114_tracking_data.json
atp_AO_Open_year_2020_MS115_tracking_data.json
atp_AO_Open_year_2020_MS116_tracking_data.json
atp_AO_Open_year_2020_MS117_tracking_data.json
atp_AO_Open_year_2020_MS118_tracking_data.json
atp_AO_Open_year_2020_MS120_tracking_data.json
atp_AO_Open_year_2020_MS121_tracking_data.json
atp_AO_Open_year_2020_MS122_tracking_data.json
atp_AO_Open_y

## Save all players

In [17]:

catalogue_df = pd.read_csv("players_matches_catalogue/catalogue_all_matches_available.csv")

player_id_mapping = {}

for index, row in catalogue_df.iterrows():
    player1_name = row['player1']
    player2_name = row['player2']
    player1_id = row['player1_id']
    player2_id = row['player2_id']
    tournament = row['tournament']

    player_id_mapping.setdefault(tournament, {})
    player_id_mapping[tournament][player1_name] = player1_id
    player_id_mapping[tournament][player2_name] = player2_id

def get_player_id(player_name, tournament):
    if tournament in player_id_mapping:
        return player_id_mapping[tournament].get(player_name, None)
    else:
        return None

player_name = 'R.NADAL'
tournament = 'roland_garros'
player_id = get_player_id(player_name, tournament)
print("Player ID for", player_name, "in", tournament, ":", player_id)


Player ID for R.NADAL in roland_garros : 7792


### Fix catalogue names

In [18]:
catalogue_path = "players_matches_catalogue/catalogue_all_matches_available.csv"
catalogue_df = pd.read_csv(catalogue_path)


In [19]:
if 'player1' in catalogue_df.columns and 'player2' in catalogue_df.columns:
    catalogue_df['player1'] = catalogue_df['player1'].apply(normalize_name)
    catalogue_df['player2'] = catalogue_df['player2'].apply(normalize_name)

catalogue_df.to_csv(catalogue_path, index=False)

saved_catalogue_df = pd.read_csv(catalogue_path)
saved_catalogue_df.head(5)


,year,player1,player2,player1_id,player2_id,player1_country,player2_country,player1_seed,player2_seed,court_name,court_id,num_sets_completed,match_type,match_status,match_id,tournament,league
0,2020,R.NADAL,H.DELLIEN,ATPN409,ATPDA31,ESP,BOL,1.0,NaN,Rod Laver Arena,1,3,Men's Singles,C,2020_MS101,australian_open,atp
1,2020,F.DELBONIS,J.SOUSA,ATPD874,ATPSH90,ARG,POR,NaN,NaN,Court 19,21,3,Men's Singles,C,2020_MS102,australian_open,atp
2,2020,C.EUBANKS,P.GOJOWCZYK,ATPE865,ATPG967,USA,GER,NaN,NaN,Court 5,7,4,Men's Singles,C,2020_MS103,australian_open,atp
3,2020,J.KOVALIK,P.CARRENOBUSTA,ATPKC04,ATPCD85,SVK,ESP,NaN,27.0,Court 13,15,4,Men's Singles,C,2020_MS104,australian_open,atp
4,2020,N.KYRGIOS,L.SONEGO,ATPKE17,ATPSU87,AUS,ITA,23.0,NaN,Melbourne Arena,3,3,Men's Singles,C,2020_MS105,australian_open,atp


In [20]:
name_replacements = {
    'JM.DELPOTRO': 'J.DELPOTRO',
    'JL.STRUFF': 'J.STRUFF',
    'JW.TSONGA' : 'J.TSONGA',
    'PH.HERBERT': 'P.HERBERT',
    'DE.GALAN': 'D.GALAN',
    'AK.SCHMIEDLOVA': 'A.SCHMIEDLOVA',
    'C.SUÁREZNAVARRO': 'C.SUAREZNAVARRO',
    'PM.TIG': 'P.TIG',
    'F.AUGER-ALIASSIME': 'F.AUGERALIASSIME',
    'JI.LONDERO': 'J.LONDERO',
    'CH.TSENG': 'C.TSENG',
    'JP.VARILLAS': 'J.VARILLAS',
    'TM.ETCHEVERRY': 'T.ETCHEVERRY',
    'JJ.WOLF': 'J.WOLF'
}

for old_name, new_name in name_replacements.items():
    catalogue_df.loc[catalogue_df['player1'] == old_name, 'player1'] = new_name
    catalogue_df.loc[catalogue_df['player2'] == old_name, 'player2'] = new_name

catalogue_df.to_csv(catalogue_path, index=False)

saved_catalogue_df = pd.read_csv(catalogue_path)
saved_catalogue_df.head(5)


,year,player1,player2,player1_id,player2_id,player1_country,player2_country,player1_seed,player2_seed,court_name,court_id,num_sets_completed,match_type,match_status,match_id,tournament,league
0,2020,R.NADAL,H.DELLIEN,ATPN409,ATPDA31,ESP,BOL,1.0,NaN,Rod Laver Arena,1,3,Men's Singles,C,2020_MS101,australian_open,atp
1,2020,F.DELBONIS,J.SOUSA,ATPD874,ATPSH90,ARG,POR,NaN,NaN,Court 19,21,3,Men's Singles,C,2020_MS102,australian_open,atp
2,2020,C.EUBANKS,P.GOJOWCZYK,ATPE865,ATPG967,USA,GER,NaN,NaN,Court 5,7,4,Men's Singles,C,2020_MS103,australian_open,atp
3,2020,J.KOVALIK,P.CARRENOBUSTA,ATPKC04,ATPCD85,SVK,ESP,NaN,27.0,Court 13,15,4,Men's Singles,C,2020_MS104,australian_open,atp
4,2020,N.KYRGIOS,L.SONEGO,ATPKE17,ATPSU87,AUS,ITA,23.0,NaN,Melbourne Arena,3,3,Men's Singles,C,2020_MS105,australian_open,atp


In [21]:
catalogue_path = "players_matches_catalogue/catalogue_all_matches_available.csv"
catalogue_df = pd.read_csv(catalogue_path)
australian_open_ids = get_unique_player_ids(catalogue_df, 'australian_open')
roland_garros_ids = get_unique_player_ids(catalogue_df, 'roland_garros')

print("Unique player IDs from Australian Open:", australian_open_ids)
print("Unique player IDs from Roland Garros:", roland_garros_ids)

Unique player IDs from Australian Open: {'WTA316959', 'ATPQ927', 'ATPBS86', 'WTA313485', 'ATPMQ75', 'ATPTC12', 'WTA310770', 'ATPC977', 'WTA312413', 'ATPD0C1', 'ATPKE73', 'ATPHH06', 'WTA317810', 'WTA317708', 'ATPTE30', 'ATPGE33', 'ATPHG86', 'WTA319526', 'WTA315616', 'WTA312001', 'ATPKI95', 'WTA314429', 'WTA322534', 'ATPKB95', 'WTA325725', 'ATPRH16', 'WTA313968', 'ATPCG33', 'WTA315427', 'ATPWB32', 'ATPGA36', 'WTA320728', 'ATPD643', 'ATPDA31', 'ATPPL56', 'ATPMD56', 'WTA315696', 'ATPCE77', 'ATPRE44', 'ATPD923', 'ATPM0CI', 'WTA326376', 'ATPKE29', 'ATPKF17', 'ATPJ553', 'WTA329627', 'WTA318516', 'WTA328896', 'ATPV836', 'WTA327793', 'ATPGJ37', 'WTA110536', 'ATPS0IA', 'ATPW09G', 'ATPS0H2', 'ATPH756', 'WTA316159', 'ATPR975', 'ATPTA12', 'WTA323942', 'ATPEA24', 'WTA311779', 'WTA310775', 'ATPM0NI', 'ATPV708', 'WTA320313', 'WTA328120', 'WTA317414', 'WTA319489', 'ATPRH24', 'ATPG628', 'WTA314584', 'ATPH09P', 'WTA327573', 'ATPML57', 'WTA315030', 'WTA323802', 'WTA310440', 'ATPKC56', 'WTA310926', 'ATPBK2

In [22]:
def create_players_csv(catalogue_df, players_csv_path):
    australian_open_ids = get_unique_player_ids(catalogue_df, 'australian_open')
    roland_garros_ids = get_unique_player_ids(catalogue_df, 'roland_garros')
    
    players_data = []

    for player_id_ao in australian_open_ids:
        player1_entry = catalogue_df[catalogue_df['player1_id'] == player_id_ao]
        player2_entry = catalogue_df[catalogue_df['player2_id'] == player_id_ao]

        # If player_entry is not empty, add player1 information
        if not player1_entry.empty:
            player1_entry = player1_entry.iloc[0]
            player_name = player1_entry['player1']
            surname = normalize_name(player_name)
            player_index = next((index for index, player in enumerate(players_data) if player['player_name'] == surname), None)
            if player_index is not None:
                players_data[player_index]['player_id_ao'] = player_id_ao
                players_data[player_index]['league'] = player1_entry['league']
            else:
                players_data.append({'player_name': surname, 'player_id_ao': player_id_ao, 'league': player1_entry['league']})

        if not player2_entry.empty:
            player2_entry = player2_entry.iloc[0]
            player_name = player2_entry['player2']
            surname = normalize_name(player_name)
            
            player_index = next((index for index, player in enumerate(players_data) if player['player_name'] == surname), None)
            if player_index is not None:
                players_data[player_index]['player_id_ao'] = player_id_ao
                players_data[player_index]['league'] = player2_entry['league']
            else:
                players_data.append({'player_name': surname, 'player_id_ao': player_id_ao, 'league': player2_entry['league']})

    for player_id_rg in roland_garros_ids:
        player1_entry = catalogue_df[catalogue_df['player1_id'] == player_id_rg]
        player2_entry = catalogue_df[catalogue_df['player2_id'] == player_id_rg]

        if not player1_entry.empty:
            player1_entry = player1_entry.iloc[0]
            player_name = player1_entry['player1']
            surname = normalize_name(player_name)
            
            player_index = next((index for index, player in enumerate(players_data) if player['player_name'] == surname), None)
            if player_index is not None:
                players_data[player_index]['player_id_rg'] = player_id_rg
                players_data[player_index]['league'] = player1_entry['league']
                players_data[player_index]['country'] = player1_entry['player1_country']
            else:
                # If player does not exist, add a new entry
                print("player1_entry", player1_entry)
                players_data.append({'player_name': surname, 'player_id_rg': int(player_id_rg), 'league': player1_entry['league'], 'country': player1_entry['player1_country']})

        if not player2_entry.empty:
            player2_entry = player2_entry.iloc[0]
            player_name = player2_entry['player2']
            surname = normalize_name(player_name)
            # Check if the player already exists in the list
            player_index = next((index for index, player in enumerate(players_data) if player['player_name'] == surname), None)
            if player_index is not None:
                # If player exists, update the entry with additional information
                players_data[player_index]['player_id_rg'] = player_id_rg
                players_data[player_index]['league'] = player2_entry['league']
                players_data[player_index]['country'] = player2_entry['player2_country']
            else:
                # If player does not exist, add a new entry
                players_data.append({'player_name': surname, 'player_id_rg': int(player_id_rg), 'league': player2_entry['league'], 'country': player2_entry['player2_country']})

    
    players_data = sorted(players_data, key=lambda x: x['player_name'])

    players_df = pd.DataFrame(players_data)

    players_df.to_csv(players_csv_path, index=False)

catalogue_path = "players_matches_catalogue/catalogue_all_matches_available.csv"
players_csv_path = "players.csv"
catalogue_df = pd.read_csv(catalogue_path)
create_players_csv(catalogue_df, players_csv_path)


player1_entry year                                   2020
player1                          K.ZAVATSKA
player2                           K.BERTENS
player1_id                            38594
player2_id                            17861
player1_country                         UKR
player2_country                         NED
player1_seed                            NaN
player2_seed                            5.0
court_name            Court Suzanne LENGLEN
court_id                                  2
num_sets_completed                        3
match_type                  Women's Singles
match_status                              C
match_id                              SD079
tournament                    roland_garros
league                                  wta
Name: 1727, dtype: object
player1_entry year                                     2019
player1                              M.KLIZAN
player2                             L.POUILLE
player1_id                              15883
player2_id    

In [23]:
players_csv_path = "players.csv"
player_df = pd.read_csv(players_csv_path)
player_df.head()

,player_name,player_id_ao,league,player_id_rg,country
0,A.ANISIMOVA,WTA326384,wta,40607.0,USA
1,A.BALAZS,ATPBD80,atp,15374.0,HUN
2,A.BARTY,WTA318033,wta,27902.0,AUS
3,A.BEDENE,ATPBH09,atp,19666.0,SLO
4,A.BLINKOVA,WTA324267,wta,33614.0,RUS


In [24]:
players_csv_path = "players.csv"
player_df = pd.read_csv(players_csv_path)

player_df['player_id_rg'] = player_df['player_id_rg'].fillna(0)
player_df['player_id_rg'] = player_df['player_id_rg'].astype(int)

if 'player_id_ao' in player_df.columns:
    player_df['player_id_atp'] = player_df['player_id_ao'].str.replace('ATP', '', 1).str.replace('WTA', '', 1)

player_df.to_csv(players_csv_path, index=False)

saved_player_df = pd.read_csv(players_csv_path)
print("Updated DataFrame saved to " + players_csv_path)
print(saved_player_df)

Updated DataFrame saved to players.csv
             player_name player_id_ao league  player_id_rg country  \
0            A.ANISIMOVA    WTA326384    wta         40607     USA   
1               A.BALAZS      ATPBD80    atp         15374     HUN   
2                A.BARTY    WTA318033    wta         27902     AUS   
3               A.BEDENE      ATPBH09    atp         19666     SLO   
4             A.BLINKOVA    WTA324267    wta         33614     RUS   
5               A.BOGDAN    WTA315427    wta         19858     ROU   
6              A.BOLSOVA    WTA320735    wta             0     NaN   
7                 A.BOLT      ATPBI81    atp             0     NaN   
8               A.BONDAR    WTA322942    wta         33616     HUN   
9               A.BUBLIK      ATPBK92    atp         29098     KAZ   
10              A.CAZAUX      ATPC0H0    atp         43952     FRA   
11              A.CORNET    WTA312121    wta         13263     FRA   
12    A.DAVIDOVICHFOKINA      ATPDH50    atp       

### Fixes players.csv 

In [25]:
league = 'atp'


players_info = get_all_players_info('', league)
players_info_df = get_all_players_info_df('', league)

print(players_info_df)
players_with_miss_id = players_info_df['player_name'][players_info_df['player_id_atp'].isna()]

# atp_ids = [player_info[0].replace('ATP', '', 1) for player_info in players_info if player_info[0] != '']
players_with_miss_id = np.array(players_with_miss_id)


player_name A.BALAZS
player_id 15374
player_name A.BEDENE
player_id 19666
player_name A.BOLT
player_id 0
player_name A.BUBLIK
player_id 29098
player_name A.CAZAUX
player_id 43952
player_name A.DAVIDOVICHFOKINA
player_id 39036
player_name A.DEMINAUR
player_id 36276
player_name A.FILS
player_id 47762
player_name A.GIANNESSI
player_id 18839
player_name A.HARRIS
player_id 0
player_name A.HOANG
player_id 27071
player_name A.KARATSEV
player_id 24955
player_name A.KOVACEVIC
player_id 41293
player_name A.KUZNETSOV
player_id 19748
player_name A.MANNARINO
player_id 13859
player_name A.MARTIN
player_id 0
player_name A.MICHELSEN
player_id 49300
player_name A.MOLCAN
player_id 33524
player_name A.MULLER
player_id 32341
player_name A.MURRAY
player_id 10216
player_name A.POPYRIN
player_id 36373
player_name A.RAMOS-VINOLAS
player_id 14065
player_name A.RINDERKNECH
player_id 29969
player_name A.RUBLEV
player_id 31132
player_name A.SEPPI
player_id 0
player_name A.SHEVCHENKO
player_id 42494
player_name A.

In [26]:
players_with_miss_id

array(['A.GIANNESSI', 'A.HOANG', 'A.KUZNETSOV', 'B.FRATANGELO', 'B.GOJO',
       'C.UGOCARABELLI', 'D.ISTOMIN', 'F.AGAMENONE', 'F.MELIGENIALVES',
       'G.BLANCANEAUX', 'GA.OLIVIERI', 'J.DELPOTRO', 'J.RODIONOV',
       'J.SOCK', 'L.NARDI', 'M.BOURGUE', 'M.GUINARD', 'M.KLIZAN',
       'S.BOLELLI', 'SF.RODRIGUEZTAVERNA', 'T.FABBIANO', 'Y.MADEN',
       'Z.KOLAR'], dtype=object)

In [27]:
players_csv_path = "players.csv"
players_df = pd.read_csv(players_csv_path)

player_ids_missing = {
    "A.GIANNESSI": "G983",
    "A.HOANG": "HA71",
    "A.KUZNETSOV": "KB54",
    "B.FRATANGELO": "F811",
    "B.GOJO": "GH92",
    "C.UGOCARABELLI": "U182",
    "CH.TSENG": "T0AP",
    "D.ISTOMIN": "I165",
    "F.AGAMENONE": "AA27",
    "G.BLANCANEAUX": "BU54",
    "GA.OLIVIERI": "O660",
    "J.DELPOTRO": "D683",
    "J.RODIONOV": "R09X",
    "J.SOCK": "SM25",
    "JP.VARILLAS": "V836",
    "M.BOURGUE": "BK19",
    "M.GUINARD": "GH33",
    "M.KLIZAN": "K966",
    "S.BOLELLI": "BA98",
    "SF.RODRIGUEZTAVERNA": "RH59",
    "T.FABBIANO": "F586",
    "TM.ETCHEVERRY": "EA24",
    "Y.MADEN": "MH16",
    "Z.KOLAR": "KH56",
    "F.MELIGENIALVES": "MW75",
    "L.NARDI": "N0BG"
}

for player_name, player_id in player_ids_missing.items():
    players_df.loc[players_df["player_name"] == player_name, "player_id_atp"] = player_id

players_df.to_csv(players_csv_path, index=False)

saved_players_df = pd.read_csv(players_csv_path)
print("Updated DataFrame saved to " + players_csv_path)
print(saved_players_df)

Updated DataFrame saved to players.csv
             player_name player_id_ao league  player_id_rg country  \
0            A.ANISIMOVA    WTA326384    wta         40607     USA   
1               A.BALAZS      ATPBD80    atp         15374     HUN   
2                A.BARTY    WTA318033    wta         27902     AUS   
3               A.BEDENE      ATPBH09    atp         19666     SLO   
4             A.BLINKOVA    WTA324267    wta         33614     RUS   
5               A.BOGDAN    WTA315427    wta         19858     ROU   
6              A.BOLSOVA    WTA320735    wta             0     NaN   
7                 A.BOLT      ATPBI81    atp             0     NaN   
8               A.BONDAR    WTA322942    wta         33616     HUN   
9               A.BUBLIK      ATPBK92    atp         29098     KAZ   
10              A.CAZAUX      ATPC0H0    atp         43952     FRA   
11              A.CORNET    WTA312121    wta         13263     FRA   
12    A.DAVIDOVICHFOKINA      ATPDH50    atp       

In [28]:
players_info_df = get_all_players_info_df('', league)
players_with_miss_id = players_info_df['player_name'][players_info_df['player_id_atp'].isna()]

players_with_miss_id = np.array(players_with_miss_id)
if not any(player is None for player in players_with_miss_id):
    print("No missing ATP IDs found.")
else:
    print("Missing ATP IDs found:", players_with_miss_id)

    print("Use this 'API' to find the PlayerId and add it to the players.csv in player_id_atp: https://www.atptour.com/en/-/www/site-search/SOCK/")
    raise ValueError("One or more players have a missing ID -> Players with missing ATP id: " + str(players_with_miss_id))

print("Players with missing atp id: ",  players_with_miss_id)


No missing ATP IDs found.
Players with missing atp id:  []
